In [ ]:
# ============================================================
# CHAPTER 6 - EXERCISE 1
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

# ------------------------------------------------------------
# 1. Load the diabetes dataset
# ------------------------------------------------------------

# This opens a file-selection window.
# Select your diabetes.csv file.

try:
    from google.colab import files

    uploaded = files.upload()

    if not uploaded:
        raise ValueError("No file was uploaded.")

    filename = next(iter(uploaded))
    df = pd.read_csv(filename)

except ImportError:
    # If you are not using Google Colab,
    # enter the CSV filename/path here.
    filename = input("Enter the path/name of your diabetes CSV file: ")
    df = pd.read_csv(filename)


# ------------------------------------------------------------
# 2. Check the dataset
# ------------------------------------------------------------

print("\nDataset loaded successfully!")
print("File:", filename)
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 3. Separate features and target
# ------------------------------------------------------------

# Chapter 6 diabetes dataset:
# 8 feature columns + 1 binary Outcome column

if "Outcome" in df.columns:
    X = df.drop("Outcome", axis=1)
    y = df["Outcome"]
else:
    # If Outcome is not named explicitly,
    # use the last column as the binary target.
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]


print("\nNumber of features:", X.shape[1])
print("Target column:", y.name)

print("\nClass distribution:")
print(y.value_counts())


# ------------------------------------------------------------
# 4. Split data into training and testing sets
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=5
)


# ------------------------------------------------------------
# 5. Standardize the features
# ------------------------------------------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ------------------------------------------------------------
# 6. Train Logistic Regression model
# ------------------------------------------------------------

model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)


# ------------------------------------------------------------
# 7. Get predicted probabilities
# ------------------------------------------------------------

# We need probabilities because Exercise 1
# changes the classification threshold.

y_probability = model.predict_proba(X_test_scaled)[:, 1]

# ROC-AUC does not depend on the classification threshold.
roc_auc = roc_auc_score(y_test, y_probability)


# ------------------------------------------------------------
# 8. Evaluation function
# ------------------------------------------------------------

def evaluate_threshold(threshold):

    # Convert probability into 0/1 prediction
    y_pred = (y_probability >= threshold).astype(int)

    cm = confusion_matrix(y_test, y_pred)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )
    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )
    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    return {
        "threshold": threshold,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "confusion_matrix": cm
    }


# ------------------------------------------------------------
# 9. Evaluate threshold 0.5 and threshold 0.3
# ------------------------------------------------------------

result_05 = evaluate_threshold(0.5)
result_03 = evaluate_threshold(0.3)


# ------------------------------------------------------------
# 10. Print results
# ------------------------------------------------------------

def print_result(result):

    threshold = result["threshold"]
    cm = result["confusion_matrix"]

    tn, fp, fn, tp = cm.ravel()

    print("\n" + "=" * 50)
    print(f"THRESHOLD = {threshold}")
    print("=" * 50)

    print(f"Accuracy : {result['accuracy']:.4f}")
    print(f"Precision: {result['precision']:.4f}")
    print(f"Recall   : {result['recall']:.4f}")
    print(f"F1-score : {result['f1']:.4f}")
    print(f"ROC-AUC  : {result['roc_auc']:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nConfusion Matrix Details:")
    print(f"True Negative  (TN): {tn}")
    print(f"False Positive (FP): {fp}")
    print(f"False Negative (FN): {fn}")
    print(f"True Positive  (TP): {tp}")


print_result(result_05)
print_result(result_03)


# ------------------------------------------------------------
# 11. Compare 0.5 vs 0.3
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("COMPARISON: THRESHOLD 0.5 VS 0.3")
print("=" * 65)

print(
    f"{'Metric':<15}"
    f"{'Threshold 0.5':>18}"
    f"{'Threshold 0.3':>18}"
    f"{'Change':>14}"
)

print("-" * 65)

metrics = [
    ("Accuracy", "accuracy"),
    ("Precision", "precision"),
    ("Recall", "recall"),
    ("F1-score", "f1"),
    ("ROC-AUC", "roc_auc")
]

for display_name, key in metrics:

    value_05 = result_05[key]
    value_03 = result_03[key]
    change = value_03 - value_05

    print(
        f"{display_name:<15}"
        f"{value_05:>18.4f}"
        f"{value_03:>18.4f}"
        f"{change:>+14.4f}"
    )


# ------------------------------------------------------------
# 12. Compare confusion matrices
# ------------------------------------------------------------

cm_05 = result_05["confusion_matrix"]
cm_03 = result_03["confusion_matrix"]

print("\n" + "=" * 65)
print("CONFUSION MATRIX COMPARISON")
print("=" * 65)

print("\nThreshold 0.5:")
print(cm_05)

print("\nThreshold 0.3:")
print(cm_03)

print("\nExercise 1 completed.")
print("Thresholds compared: 0.5 and 0.3")


Dataset loaded successfully!
File: C:\Users\HP\Desktop\logistic-Regression\diabetes.csv
Shape: (768, 9)

Columns:
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

Number of features: 8
Target column: Outcome

Class distribution:
Outcome
0    500
1    268
Name: count, dtype: int64

THRESHOLD = 0.5
Accuracy : 0.7987
Precision: 0.7447
Recall   : 0.6481
F1-score : 0.6931
ROC-AUC  : 0.8596

Confusion Matrix:
[[88 12]
 [19 35]]

Confusion Matrix Details:
True Negative  (TN): 88
False Positive (FP): 12
False Negative (FN): 19
True Positive  (TP): 35

THRESHOLD = 0.3
Accuracy : 0.7532
Precision: 0.6143
Recall   : 0.7963
F1-score : 0.6935
ROC-AUC  : 0.8596

Confusion Matrix:
[[73 27]
 [11 43]]

Confusion Matrix Details:
True Negative  (TN): 73
False Positive (FP): 27
False Negative (FN): 11
True Positive  (TP): 43

COMPARISON: THRESHOLD 0.5 VS 0.3
Metric              Threshold 0.5     Threshold 0.3        Change
-----